# 02 — Phase-1 Pilot Verification & Automated GO / NO-GO Scorecard

**Scope**: `When2Speak-Pilot` (400 turns) + `pm_official_8` + 8 mini PageMantle-Ext tasks (4 action + 4 traps) + 1 mini online DB-CSE wave (8 stream tasks).  
**Estimated Budget**: ~\$5–\$15 total API spend (cached on Drive so re-runs cost \$0).

In [1]:
# === CELL 0: Mount Google Drive & Configure Persistent Workspace ===
import os
from pathlib import Path

IN_COLAB = "COLAB_RELEASE_TAG" in os.environ or Path("/content").exists()
if IN_COLAB:
    from google.colab import drive
    drive.mount("/content/drive", force_remount=False)
    DRIVE_ROOT = Path("/content/drive/MyDrive/when2speak_research")
else:
    DRIVE_ROOT = Path("./local_drive_mirror").resolve()

for sub in [
    "secrets",
    "config_cache",
    "datasets/pagemantle",
    "datasets/when2speak",
    "datasets/pagemantle_ext",
    "caches",
    "runs",
    "paper_artifacts/tables",
    "paper_artifacts/figures",
]:
    (DRIVE_ROOT / sub).mkdir(parents=True, exist_ok=True)

os.environ["DIAL_SKILL_DRIVE_ROOT"] = str(DRIVE_ROOT)
print("Persistent DRIVE_ROOT:", DRIVE_ROOT)


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Persistent DRIVE_ROOT: /content/drive/MyDrive/when2speak_research


In [2]:
# === CELL 1: Sync GitHub Repo (git clone / git pull) & Hot-Reload ===
REPO_URL = "https://github.com/BoldHu/multiagent-speak.git"  # Edit once with your repo URL
REPO_DIR = Path("/content/when2speak") if IN_COLAB else Path(".").resolve()

if IN_COLAB:
    if not REPO_DIR.exists():
        !git clone {REPO_URL} {REPO_DIR}
    else:
        !git -C {REPO_DIR} pull --ff-only
    %cd {REPO_DIR}
    !pip install -q -e . --no-deps

import importlib, sys, types

# Colab Python 3.12+/3.13 removed stdlib `imp`, which older IPython autoreload still imports.
if "imp" not in sys.modules:
    try:
        import imp  # noqa: F401
    except ModuleNotFoundError:
        _imp_shim = types.ModuleType("imp")
        _imp_shim.reload = importlib.reload
        sys.modules["imp"] = _imp_shim

try:
    _ip = get_ipython()
    if _ip is not None:
        _ip.run_line_magic("load_ext", "autoreload")
        _ip.run_line_magic("autoreload", "2")
except Exception as _exc:
    print(f"Skipping autoreload ({_exc}); restart runtime after git pull if needed.")


Already up to date.
/content/when2speak
  Installing build dependencies ... done
  Checking if build backend supports build_editable ... done
  Getting requirements to build editable ... done
  Preparing editable metadata (pyproject.toml) ... done
  Building editable for dial-skill (pyproject.toml) ... done


In [3]:
# === CELL 2: Bootstrap Secrets & Resolved Model Config (Never Prints Raw Keys) ===
import json
from dial_skill.utils.secrets import describe_secret_sources
from dial_skill.llm.preflight import load_resolved_models, resolve_model_for_role
from dial_skill.config import load_models_config, load_dial_skill_config, load_experiments_config

print("Secret lookup diagnostics:", json.dumps(describe_secret_sources(drive_root=DRIVE_ROOT), indent=2))
MODELS_CFG = load_models_config()
DIAL_CFG = load_dial_skill_config()
EXP_CFG = load_experiments_config()
RESOLVED_MODELS = load_resolved_models(drive_root=DRIVE_ROOT)
print("Resolved models:", RESOLVED_MODELS.model_dump() if RESOLVED_MODELS else "Not yet generated (run Notebook 00)")


Secret lookup diagnostics: {
  "jev": {
    "status": "found",
    "source": "drive:JEV_API_KEY.txt",
    "masked": "apik...c3aa"
  },
  "openai": {
    "status": "found",
    "source": "drive:OPENAI_KEY.txt",
    "masked": "sk-p...yN8A"
  }
}
Resolved models: {'timestamp_utc': '2026-11-20T13:25:11.705978+00:00', 'luna': 'gpt-4.1-mini', 'terra': 'gpt-4.1', 'sol': 'gpt-6.1-sol', 'jev_model': 'jev-latest', 'jev_healthy': True, 'triage_backend': 'jev', 'available_openai_models_count': 99, 'notes': ["Probe failed for luna candidate 'gpt-6-luna': BadRequestError", "Probe failed for luna candidate 'gpt-5.6-luna': BadRequestError", "Probe failed for terra candidate 'gpt-6.1-terra': BadRequestError", "Probe failed for terra candidate 'gpt-5.6-terra': BadRequestError"]}


## Step 1: Cost Estimate & Run Configuration (`MODE = 'pilot'`)

In [4]:
from datetime import datetime, timezone
from dial_skill.eval.runners import estimate_run_cost_usd
from dial_skill.utils.checkpoint import save_run_metadata

MODE = "pilot"
est = estimate_run_cost_usd(mode=MODE, include_pagemantle=True, include_ablations=False)
print("Estimated Pilot API Calls & Cost:", json.dumps(est, indent=2))

RUN_DIR = DRIVE_ROOT / "runs" / f"pilot_{datetime.now(timezone.utc).strftime('%Y%m%d')}"
save_run_metadata(RUN_DIR, {"mode": MODE, "estimate": est, "dial_cfg": DIAL_CFG.model_dump()})
print("Run outputs will persist to:", RUN_DIR)


Estimated Pilot API Calls & Cost: {
  "mode": "pilot",
  "estimated_api_calls": 1072,
  "estimated_when2speak_usd": 0.88,
  "estimated_pagemantle_usd": 2.3,
  "estimated_ablations_usd": 0.0,
  "estimated_total_usd": 3.18
}
Run outputs will persist to: /content/drive/MyDrive/when2speak_research/runs/pilot_20261120


## Step 2: When2Speak-Pilot (400 turns) — `DIAL-Skill (Luna)` vs `B0 Direct Chat (Luna)`

In [5]:
from dial_skill.baselines.direct_chat import DirectChatBaseline
from dial_skill.benchmarks.when2speak import load_when2speak_split
from dial_skill.core.pipeline import DialSkillAgent
from dial_skill.core.skill_store import SkillStore
from dial_skill.eval.metrics import compute_when2speak_metrics, mcnemar_test
from dial_skill.eval.runners import run_when2speak_eval
from dial_skill.llm.jev_client import TriageClient
from dial_skill.llm.openai_client import OpenAIClient
from dial_skill.utils.cache import SQLiteAPICache

cache = SQLiteAPICache(DRIVE_ROOT / "caches" / "api_cache.sqlite")
oai = OpenAIClient(models_cfg=MODELS_CFG, cache=cache)
luna_model = resolve_model_for_role("gate", MODELS_CFG, RESOLVED_MODELS)
terra_model = resolve_model_for_role("executor", MODELS_CFG, RESOLVED_MODELS)
triage_backend = RESOLVED_MODELS.triage_backend if RESOLVED_MODELS else MODELS_CFG.roles.triage_backend
triage_client = TriageClient(
    models_cfg=MODELS_CFG, cache=cache, openai_client=oai, luna_model=luna_model, backend=triage_backend
)

w2s_pilot = load_when2speak_split("pilot", local_dir=DRIVE_ROOT / "datasets" / "when2speak", seed=EXP_CFG["seed"])
dial_agent = DialSkillAgent(
    triage_client=triage_client,
    openai_client=oai,
    gate_model=luna_model,
    executor_model=terra_model,
    skill_store=SkillStore.from_json_file(),
    cfg=DIAL_CFG,
    method_name="dial_skill_luna",
)
direct_agent = DirectChatBaseline(openai_client=oai, model=luna_model)

w2s_dial_eps = await run_when2speak_eval(
    agent=dial_agent, turns=w2s_pilot, output_jsonl=RUN_DIR / "predictions_when2speak_dial.jsonl", resume=True
)
w2s_direct_eps = await run_when2speak_eval(
    agent=direct_agent, turns=w2s_pilot, output_jsonl=RUN_DIR / "predictions_when2speak_direct.jsonl", resume=True
)

y_true = [t.label_speak for t in w2s_pilot]
w2s_dial_m = compute_when2speak_metrics(y_true, w2s_dial_eps)
w2s_direct_m = compute_when2speak_metrics(y_true, w2s_direct_eps)
mc = mcnemar_test(y_true, [e.gate.binary_speak for e in w2s_dial_eps], [e.gate.binary_speak for e in w2s_direct_eps])
print("When2Speak-Pilot DIAL-Skill:", json.dumps(w2s_dial_m, indent=2))
print("When2Speak-Pilot Direct Chat:", json.dumps(w2s_direct_m, indent=2))
print("McNemar Test:", mc)


When2Speak-Pilot DIAL-Skill: {
  "n_samples": 400,
  "tp": 9,
  "tn": 324,
  "fp": 25,
  "fn": 42,
  "macro_f1": 0.559029,
  "macro_f1_ci95": [
    0.501396,
    0.62485
  ],
  "speak_f1": 0.211765,
  "speak_precision": 0.264706,
  "speak_recall": 0.176471,
  "silent_f1": 0.906294,
  "fir": 0.071633,
  "mir": 0.823529,
  "auroc": 0.633912,
  "auprc": 0.204409,
  "ece": 0.166182,
  "margin_gap": 0.133499,
  "token_reduction_pct": -165.04,
  "short_circuit_rate": 0.005,
  "total_cost_usd": 1.56828,
  "cost_per_1k_turns_usd": 3.9207
}
When2Speak-Pilot Direct Chat: {
  "n_samples": 400,
  "tp": 36,
  "tn": 141,
  "fp": 208,
  "fn": 15,
  "macro_f1": 0.401242,
  "macro_f1_ci95": [
    0.355797,
    0.449898
  ],
  "speak_f1": 0.244068,
  "speak_precision": 0.147541,
  "speak_recall": 0.705882,
  "silent_f1": 0.558416,
  "fir": 0.595989,
  "mir": 0.294118,
  "auroc": 0.554947,
  "auprc": 0.141647,
  "ece": 0.510074,
  "margin_gap": 0.219788,
  "token_reduction_pct": 0.0,
  "short_circuit_rat

## Step 3: PageMantle Pilot (`pm_official_8` + 8 mini traps/actions) + 1 Mini DB-CSE Wave

In [6]:
from dial_skill.benchmarks.pagemantle_env import PageMantleEnv
from dial_skill.eval.judge import BlindedDualJudge
from dial_skill.eval.metrics import compute_pagemantle_metrics
from dial_skill.eval.runners import run_online_evolution_stream, run_pagemantle_eval
from dial_skill.evolution.db_cse import DBCSEUpdater
from dial_skill.schemas import PageMantleTask

def _load_pm(name: str):
    p = DRIVE_ROOT / "datasets" / "pagemantle_ext" / f"{name}.jsonl"
    return [PageMantleTask.model_validate_json(line) for line in p.read_text(encoding="utf-8").splitlines() if line.strip()]

pm_official = _load_pm("pm_official_8")
pm_stream_w1 = [t for t in _load_pm("pm_online_stream_24") if t.wave == 1]
pm_eval_all = _load_pm("pm_eval_32")
pm_mini_eval = [t for t in pm_eval_all if t.expected_action == "INTERVENE"][:4] + [
    t for t in pm_eval_all if t.expected_action == "SILENT"
][:4]
pm_pilot_tasks = pm_official + pm_mini_eval

env = PageMantleEnv(dataset_dir=DRIVE_ROOT / "datasets" / "pagemantle")
env.ensure_extracted()
judge = BlindedDualJudge(openai_client=oai, primary_judge_model=terra_model)

pm_dial_eps = await run_pagemantle_eval(
    agent=dial_agent, tasks=pm_pilot_tasks, env=env, output_jsonl=RUN_DIR / "predictions_pm_dial.jsonl", resume=True
)
pm_direct_eps = await run_pagemantle_eval(
    agent=DirectChatBaseline(openai_client=oai, model=terra_model),
    tasks=pm_pilot_tasks,
    env=env,
    output_jsonl=RUN_DIR / "predictions_pm_direct.jsonl",
    resume=True,
)
pm_dial_scores = [await judge.grade_pagemantle_episode(t, e) for t, e in zip(pm_pilot_tasks, pm_dial_eps)]
pm_direct_scores = [await judge.grade_pagemantle_episode(t, e) for t, e in zip(pm_pilot_tasks, pm_direct_eps)]

pm_dial_m = compute_pagemantle_metrics(pm_pilot_tasks, pm_dial_eps, pm_dial_scores)
pm_direct_m = compute_pagemantle_metrics(pm_pilot_tasks, pm_direct_eps, pm_direct_scores)

# Mini DB-CSE Wave 1 sanity check (strictly on pm_online_stream_24 Wave 1)
updater = DBCSEUpdater(
    skill_store=dial_agent.skill_store,
    openai_client=oai,
    critic_model=terra_model,
    cfg=DIAL_CFG,
    log_path=RUN_DIR / "skill_evolution_log.jsonl",
)
wave1_summary = await run_online_evolution_stream(
    agent=dial_agent, updater=updater, stream_tasks=pm_stream_w1, env=env, run_dir=RUN_DIR / "mini_wave1", judge=judge
)
pm_post_eps = await run_pagemantle_eval(
    agent=dial_agent, tasks=pm_mini_eval, env=env, output_jsonl=RUN_DIR / "predictions_pm_post_w1.jsonl", resume=False
)
pm_post_scores = [await judge.grade_pagemantle_episode(t, e) for t, e in zip(pm_mini_eval, pm_post_eps)]
pm_pre_mini = compute_pagemantle_metrics(pm_mini_eval, pm_dial_eps[len(pm_official):], pm_dial_scores[len(pm_official):])
pm_post_mini = compute_pagemantle_metrics(pm_mini_eval, pm_post_eps, pm_post_scores)
dbcse_delta = pm_post_mini["overall_score"] - pm_pre_mini["overall_score"]
print("Mini Wave-1 DB-CSE held-out delta:", round(dbcse_delta, 4))


Mini Wave-1 DB-CSE held-out delta: -0.2437


## Step 4: Automated Pilot GO / NO-GO Scorecard

In [7]:
from dial_skill.eval.metrics import evaluate_pilot_go_nogo

long_eps = [e for t, e in zip(w2s_pilot, w2s_dial_eps) if len(t.messages) > 15]
long_raw = sum(e.prompt_tokens_raw for e in long_eps) or 1
long_slate = sum(e.prompt_tokens_slate for e in long_eps)
long_red = (1.0 - long_slate / long_raw) if long_eps else (w2s_dial_m["token_reduction_pct"] / 100.0)

scorecard = evaluate_pilot_go_nogo(
    w2s_dial=w2s_dial_m,
    w2s_direct=w2s_direct_m,
    pm_dial=pm_dial_m,
    pm_direct=pm_direct_m,
    triage_long_chat_token_reduction=long_red,
    triage_decisive_recall=1.0 - w2s_dial_m["mir"],
    dbcse_anchor_regressions=0,
    dbcse_heldout_delta=dbcse_delta,
    thresholds=EXP_CFG.get("pilot_go_nogo"),
)
scorecard["triage_meta"] = triage_client.get_run_metadata()
scorecard["total_pilot_usd"] = round(oai.usage.total_cost_usd + triage_client.total_cost_usd, 4)

(RUN_DIR / "pilot_scorecard.json").write_text(json.dumps(scorecard, indent=2), encoding="utf-8")
print("=== PILOT VERIFICATION SCORECARD ===")
print(json.dumps(scorecard, indent=2))


=== PILOT VERIFICATION SCORECARD ===
{
  "verdict": "NO_GO",
  "passed_checks": 0,
  "total_checks": 5,
  "checks": {
    "1_when2speak_pilot_timing": false,
    "2_pagemantle_end_to_end": false,
    "3_jev_triage_compression": false,
    "4_margin_discriminability": false,
    "5_dbcse_sanity": false
  },
  "diagnostics": [
    "MIR too high: lower tau_dm (0.05 -> -0.05), theta_skip (0.15 -> 0.08), or theta_floor."
  ],
  "triage_meta": {
    "configured_triage_backend": "jev",
    "jev_calls": 2237,
    "jev_cache_hits": 626,
    "fallback_count": 0,
    "jev_circuit_open_reason": null,
    "fallback_reasons_sample": [],
    "triage_cost_usd": 1.1185
  },
  "total_pilot_usd": 2.7815
}


## Step 5: Zero-Cost Pilot Deep Diagnostics (Reads Saved Drive Artifacts, $0.00)

Inspects exact When2Speak score distributions, PageMantle per-task metrics & tool traces, and DB-CSE Wave-1 evolution logs so we can tune surgically before any re-run.

In [ ]:
import json
import numpy as np
from pathlib import Path
from dial_skill.benchmarks.when2speak import load_when2speak_split
from dial_skill.eval.metrics import compute_when2speak_metrics
from dial_skill.schemas import EpisodeResult, PageMantleTask
from dial_skill.utils.checkpoint import load_jsonl

# Auto-locate latest pilot_* run directory on Drive
pilot_dirs = sorted((DRIVE_ROOT / "runs").glob("pilot_*"))
DIAG_RUN_DIR = pilot_dirs[-1]
print("Inspecting saved Pilot Run:", DIAG_RUN_DIR)

# 1. When2Speak Deep Diagnostics & Zero-Cost Threshold Sweep
w2s_turns = load_when2speak_split("pilot", local_dir=DRIVE_ROOT / "datasets" / "when2speak", seed=EXP_CFG["seed"])
w2s_dial_rows = [EpisodeResult.model_validate(r) for r in load_jsonl(DIAG_RUN_DIR / "predictions_when2speak_dial.jsonl")]
by_id = {e.sample_id: e for e in w2s_dial_rows}
paired = [(t, by_id[t.sample_id]) for t in w2s_turns if t.sample_id in by_id]

pos_adv = [e.gate.advocate_score for t, e in paired if t.label_speak]
neg_adv = [e.gate.advocate_score for t, e in paired if not t.label_speak]
pos_skep = [e.gate.skeptic_score for t, e in paired if t.label_speak]
neg_skep = [e.gate.skeptic_score for t, e in paired if not t.label_speak]
pos_del = [e.gate.margin_delta for t, e in paired if t.label_speak]
neg_del = [e.gate.margin_delta for t, e in paired if not t.label_speak]

print("\n--- When2Speak-Pilot Stage-2 Score Distributions ---")
print(f"SPEAK  (n={len(pos_adv)}): adv={np.mean(pos_adv):.3f}±{np.std(pos_adv):.3f}, skep={np.mean(pos_skep):.3f}±{np.std(pos_skep):.3f}, delta={np.mean(pos_del):.3f}")
print(f"SILENT (n={len(neg_adv)}): adv={np.mean(neg_adv):.3f}±{np.std(neg_adv):.3f}, skep={np.mean(neg_skep):.3f}±{np.std(neg_skep):.3f}, delta={np.mean(neg_del):.3f}")
print(f"Turn length stats: min={min(len(t.messages) for t,_ in paired)}, median={np.median([len(t.messages) for t,_ in paired]):.1f}, max={max(len(t.messages) for t,_ in paired)}, >15 turns={sum(1 for t,_ in paired if len(t.messages)>15)}")
print(f"Avg raw tokens={np.mean([e.prompt_tokens_raw for _,e in paired]):.1f}, avg slate tokens={np.mean([e.prompt_tokens_slate for _,e in paired]):.1f}")

# Zero-cost grid search over (lambda, tau_dm, theta_floor) on existing Stage-2 scores
from sklearn.metrics import f1_score
yt = [int(t.label_speak) for t, _ in paired]
best = (-1.0, None)
for lam in [0.25, 0.5, 0.75, 1.0]:
    for tau in [-0.40, -0.30, -0.20, -0.10, -0.05, 0.0, 0.05]:
        for floor in [0.15, 0.20, 0.25, 0.30]:
            yp = [
                1 if (not e.gate.short_circuited and e.gate.advocate_score >= floor and (e.gate.advocate_score - lam * e.gate.skeptic_score) >= tau) else 0
                for _, e in paired
            ]
            f1 = float(f1_score(yt, yp, average="macro", zero_division=0))
            fp = sum(1 for y, p in zip(yt, yp) if y == 0 and p == 1)
            fn = sum(1 for y, p in zip(yt, yp) if y == 1 and p == 0)
            fir = fp / max(1, sum(1 - y for y in yt))
            mir = fn / max(1, sum(yt))
            if f1 > best[0]:
                best = (f1, {"lambda": lam, "tau_dm": tau, "theta_floor": floor, "macro_f1": round(f1, 4), "fir": round(fir, 4), "mir": round(mir, 4)})
print("Best zero-cost threshold point on current Stage-2 scores:", best[1])

print("\nSample 5 False Negatives (missed SPEAK turns):")
fns = [(t, e) for t, e in paired if t.label_speak and not e.gate.binary_speak][:5]
for t, e in fns:
    last = t.messages[-1]
    print(f"  [{t.sample_id}] len={len(t.messages)} adv={e.gate.advocate_score:.2f} skep={e.gate.skeptic_score:.2f} skill={e.gate.selected_skill} | {last.speaker}: {last.text[:140]!r}")

# 2. PageMantle Pilot Breakdown
def _load_pm(name: str):
    fp = DRIVE_ROOT / "datasets" / "pagemantle_ext" / f"{name}.jsonl"
    return [PageMantleTask.model_validate_json(line) for line in fp.read_text(encoding="utf-8").splitlines() if line.strip()]

pm_official = _load_pm("pm_official_8")
pm_eval_all = _load_pm("pm_eval_32")
pm_mini_eval = [t for t in pm_eval_all if t.expected_action == "INTERVENE"][:4] + [t for t in pm_eval_all if t.expected_action == "SILENT"][:4]
pm_tasks = pm_official + pm_mini_eval
pm_dial_map = {r["sample_id"]: EpisodeResult.model_validate(r) for r in load_jsonl(DIAG_RUN_DIR / "predictions_pm_dial.jsonl")}
pm_dir_map = {r["sample_id"]: EpisodeResult.model_validate(r) for r in load_jsonl(DIAG_RUN_DIR / "predictions_pm_direct.jsonl")}
pm_post_map = {r["sample_id"]: EpisodeResult.model_validate(r) for r in load_jsonl(DIAG_RUN_DIR / "predictions_pm_post_w1.jsonl")}

print("\n--- PageMantle Pilot Per-Task Comparison (16 tasks) ---")
for t in pm_tasks:
    ed = pm_dial_map.get(t.task_id)
    eb = pm_dir_map.get(t.task_id)
    ep_post = pm_post_map.get(t.task_id)
    print(
        f"  {t.task_id:15s} | cat={t.category:10s} | exp={t.expected_action:9s} ({t.ideal_tier.value:17s}) "
        f"| DIAL={ed.gate.final_tier.value if ed else 'NA':17s} (adv={ed.gate.advocate_score:.2f},skep={ed.gate.skeptic_score:.2f},tools={ed.tool_calls_used},to={ed.timed_out}) "
        f"| DIRECT={eb.gate.final_tier.value if eb else 'NA':17s} (tools={eb.tool_calls_used},to={eb.timed_out})"
        + (f" | POST_W1={ep_post.gate.final_tier.value}" if ep_post else "")
    )

# Print in-memory pm_dial_m / pm_direct_m if still in session
if "pm_dial_m" in globals() and "pm_direct_m" in globals():
    print("\nPageMantle DIAL-Skill metrics:", json.dumps(pm_dial_m, indent=2))
    print("PageMantle Direct Chat metrics:", json.dumps(pm_direct_m, indent=2))
    print("PageMantle Pre-Mini vs Post-Mini:", json.dumps({"pre": pm_pre_mini, "post": pm_post_mini}, indent=2))

# 3. DB-CSE Wave-1 Evolution Log
evo_log_path = DIAG_RUN_DIR / "skill_evolution_log.jsonl"
if evo_log_path.is_file():
    evo_rows = load_jsonl(evo_log_path)
    print(f"\n--- DB-CSE Wave-1 Evolution Log ({len(evo_rows)} entries) ---")
    for row in evo_rows:
        print(f"  {row['sample_id']:18s} | skill={row['skill_name']:10s} | fb={row['feedback_type']:20s} | rc={str(row.get('root_cause')):18s} | promoted={row['promoted']} | reject={row.get('rejection_reason')} | num={row.get('numeric_update')}")
